# T8 It2: audit of the earlier methods: fitted to the data? real gain or just more answers?

**Team review (2026-11-14):** the T7 judge was trained on the data it was meant to improve, so it fits that data; the final data will not be the
same. Every earlier method is checked the same way here.

**Protocol (locked):** official is cut by place into a TRY half (185 photos) and a TEST half (194), heights balanced (It0 split,
`results/official_split.csv`, block `train/checks/holdout.py`). This notebook never loads the TEST half. Design data = the five NUDT sets.

**Two questions per method:**
1. *Fitted?* The same method built with official in its training/design vs built on NUDT only, both scored on official TRY.
2. *Real gain or more answers?* A method that answers more photos is compared with the **plain inlier bar that answers the same number of photos**
   (`versus_frontier`): sitting on that curve means it only answers more; above it means it judges better. `answer_change` splits the gain into
   answers added (right / wrong), answers removed and wrong answers.

In [ ]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd, joblib
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
from train.checks.candidate_judge import (FEATURES_CTX, TAUS, baseline, decide, judge_new, judge_probability, ltt_tau, scan_tau, summary)
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, load_t8_table, versus_frontier
from train.checks.two_tier import null_decide, null_pvalues
RES = ROOT / "tests/Test_Iteration_8/results"
table = load_t8_table(ROOT, locked=True)          # official_test is not loaded here
SEEN = ["official_try"] + NUDT_DESIGN
assert "official_test" not in set(table.data)
table.groupby("data").photo.nunique()

## 1. The plain-bar curve (the reference)
PC: bar b on the 6 standard queries, b + 3 on all 27. Jetson: bar b, second pass, cap 10, M = 5.

In [ ]:
t0 = time.time(); FRONT = bar_frontier(table); FRONT.to_csv(RES / "it2_bar_frontier.csv", index=False); print("%.0f s" % (time.time() - t0))
for var in ["pc", "jetson"]:
    f = FRONT[FRONT["var"] == var]
    display(pd.concat({"score": f.pivot(index="bar", columns="data", values="score"), "wrong": f.pivot(index="bar", columns="data", values="wrong")}, axis=1).round(2))

**Read:** on official TRY a bar of 4 makes **0 wrong answers** and scores 54.8 (PC), +3.4 over bar 8; on NUDT the same bar makes 6-15 wrong
answers per set. So on official, *answering more is almost free*, and any method that answers more will look good there. The decision problem is
set by how many look-alike places a map has, which differs between data sets.

## 2. The T7 judge: trained with official vs on NUDT only

In [ ]:
frozen = joblib.load(ROOT / "tests/Test_Iteration_7/results/judge_t7.joblib")
frozen = frozen["model"] if isinstance(frozen, dict) else frozen
p_frozen = judge_new(frozen, table)                                   # trained on official (all 379) + NUDT 101-505: in-sample on TRY
p_nudt, _ = judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm")   # trained on the five NUDT sets only (out-of-fold on NUDT)
KW = dict(override=12, min_choice=3, wide_tau=0.95)
tau_pc = max(ltt_tau(scan_tau(table, p_nudt, TAUS, **KW), NUDT_DESIGN, 0.005), 0.9)
tau_jet = max(ltt_tau(scan_tau(table, p_nudt, TAUS, jetson=True, m=5, **KW), NUDT_DESIGN, 0.005), 0.9)
print("cuts chosen on NUDT only: PC %.3f, Jetson %.3f" % (tau_pc, tau_jet))
M = {"pc": {}, "jetson": {}}
M["pc"]["bar 8 / 11 (base)"] = baseline(table, 8, 11, min_choice=3)
M["jetson"]["Jetson today (base)"] = baseline(table, 8, 8, jetson=True, m=5)
M["pc"]["T7 judge, trained with official (frozen)"] = decide(table, p_frozen, 0.9, **KW)
M["pc"]["T7 judge, NUDT only"] = decide(table, p_nudt, tau_pc, **KW)
M["jetson"]["T7 judge, NUDT only"] = decide(table, p_nudt, tau_jet, jetson=True, m=5, **KW)
M["jetson"]["T7 judge, NUDT only + early exit 0.98"] = decide(table, p_nudt, tau_jet, jetson=True, m=5, early_tau=0.98, **KW)
M["pc"]["T7 judge, NUDT only, min 4 inliers"] = decide(table, p_nudt, tau_pc, **{**KW, "min_choice": 4})
veto = table.copy(); veto.loc[(veto.z < 1.3) & (veto.inliers < 15), "inliers"] = 0      # T6 DINO-z veto: such candidates can be neither chosen nor answered
M["pc"]["T6 z-veto on bar 8 / 11"] = baseline(veto, 8, 11, min_choice=3)
M["jetson"]["T6 z-veto on Jetson today"] = baseline(veto, 8, 8, jetson=True, m=5)
pp = null_pvalues(table, "inliers")
M["pc"]["null-calibrated inlier cut, eps 0.05 / 0.017"] = null_decide(pp, 0.05, 0.05 / 3)
table["pj"] = p_nudt
ppj = null_pvalues(table, "pj")
M["pc"]["null-calibrated judge cut, eps 0.02 / 0.007"] = null_decide(ppj, 0.02, 0.02 / 3)
rows = []
for var, ms in M.items():
    base = list(ms.values())[0]
    for name, r in ms.items():
        v = versus_frontier(r, FRONT, var).set_index("data")
        a = answer_change(base, r).set_index("data")
        for d in SEEN:
            rows.append(dict(var=var, method=name, data=d, **v.loc[d].to_dict(), **a.loc[d, ["added_right", "added_wrong", "removed", "removed_wrong"]].to_dict()))
AUDIT = pd.DataFrame(rows); AUDIT.to_csv(RES / "it2_audit.csv", index=False)
for d in SEEN:
    print("==", d); display(AUDIT[AUDIT.data == d].drop(columns="data").set_index(["var", "method"]).round(2))

**Read (official TRY):**
- *Fitted:* the T7 judge that saw official scores ~57 on TRY; the same architecture trained on NUDT only scores ~54. About **3 points of the T7
  official number came from fitting official itself** (in-sample), which supports the review.
- *More answers:* trained on NUDT only, the judge sits **on the plain-bar curve on official** (about +0.2 over the bar answering as many photos):
  there it only answers more. On NUDT it sits above the curve (+1 to +3) because it removes the bars' 1-2 wrong answers per set, but it adds a
  new wrong answer on NUDT 202 and keeps one on 606.
- The Jetson early exit costs ~1.2 points on official TRY (answers taken before the most precise query) for ~60 % less matching time.
- The T6 DINO-z veto only removes wrong answers (404, 505, 606) and costs nothing on TRY: a safety net, not a gain.
- The label-free null-calibrated cut (idea: set the cut from each set's own far candidates, so look-alike-rich maps get a stricter cut without
  labels) is safe with the inlier score but scores below today's bar, and with the judge's score it lets 1-2 wrong answers into NUDT: dropped.

## 3. Precision: other position estimates of the chosen place (answers at >= 8 inliers within 8 sigma)

In [ ]:
import glob
truth = {}
for d, folder in [("official_try", "official")] + [(s, {"nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404", "nudt505": "nudt_fresh505", "nudt606": "nudt_fresh606"}[s]) for s in NUDT_DESIGN]:
    for f in glob.glob(str(ROOT / f"tests/Test_Iteration_6/results/rich_dump/{folder}/photo_*.npz")):
        truth[(d, int(f.split("_")[-1][:-4]))] = tuple(np.load(f, allow_pickle=True)["truth"][:2])
rows = []
for (d, p), g in table.groupby(["data", "photo"], sort=False):
    g = g[g.inliers >= 3]
    if not len(g) or (d, p) not in truth: continue
    c = g.iloc[int(np.lexsort((-g.coarse.to_numpy(), -(g.inliers + g.z).to_numpy()))[0])]
    if c.inliers < 8 or c.err_sigma > 8: continue
    tx, ty = truth[(d, p)]; near = g[np.hypot(g.x - c.x, g.y - c.y) <= 2 * c.sigma]; w = near.inliers.to_numpy()
    est = {"similarity fit (today)": (c.x, c.y), "translation fit": (c.tx, c.ty) if c.inl_t3 > 0 else (c.x, c.y),
           "inlier-weighted mean of the place over queries": ((near.x * w).sum() / w.sum(), (near.y * w).sum() / w.sum()),
           "largest-hull candidate of the place": tuple(near.loc[near.hull.idxmax(), ["x", "y"]])}
    rows.append(dict(data=d, **{k: np.hypot(x - tx, y - ty) / c.sigma for k, (x, y) in est.items()}))
E = pd.DataFrame(rows)
pts = lambda v: np.select([v <= 1, v <= 2, v <= 4, v <= 8], [10, 6, 3, 0], -20).mean()
E.groupby("data").agg(pts).round(3)

**Read:** today's similarity-fit position is already the best estimate on every set (official TRY 9.81 of 10 per answer); no change here.